In [ ]:
import pandas as pd
import numpy as np
import datetime as dt
from sqlalchemy import create_engine
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
# import all the data tables into single file
customers = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/customers.csv")
employees = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/employees.csv")
inventory = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/inventory.csv")
orders = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/orders.csv")
products = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/products.csv")
promotions = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/promotions.csv")
returns = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/returns.csv")
shipping = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/shipping.csv")
vendors = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/vendors.csv")
warehouses = pd.read_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/warehouses.csv")


In [37]:
orders['profit_contribution_pct'] = ((orders['profit']/orders['profit'].sum())*100).round(2)

In [ ]:
customers['age_group'] = pd.cut(customers['age'], 
    bins= [18,25,35,45,55,65,100],
    labels=['18-25','26-35','36-45','46-55','56-65','65+'],
    right=True)

In [36]:
customers.columns

Index(['customer_id', 'customer_name', 'gender', 'age', 'city', 'state',
       'country', 'join_date', 'customer_segment', 'join_month', 'join_year',
       'age_group'],
      dtype='object')

In [ ]:
# sales analysis
sales_data = pd.merge(
    orders,
    products,
    how="right",
    on="product_id"
)

sales_data['brand'] = sales_data['brand'].str.strip().str.capitalize()
sales_data.columns
# sales by product_category
product_category_sales = sales_data.groupby(['category', 'sub_category'], as_index=False)[['sales','quantity', 'discount']].sum().sort_values('sales', ascending=False)
product_category_sales
# insights
# we find out that Electronics category has the highest sales value, with second highest total quantity solds , and the highest discount is also offer on Electronics Category
# second Home Category with third highest quantity solds, and discount offers on it
# least was books category, with 2nd last quantity solds and discount offered on this 
sales_by_brand = sales_data.groupby(['brand'], as_index=False)[['sales', 'discount', 'quantity','profit_x','profit_y','cost_price']].sum().sort_values("sales" , ascending=False)
sales_by_brand

# top 5 sales by brand = borosil  , MI, Syska, Noise, Maybeline
# bottom 5 sales by brand is - cannnon, lg, nike, apple,cello
# top 5 brands by heavy discount - lenovo, ikea, puma, adidas, mi
# bottom 5 brands with least discount -  lg, nike, sony, apple, casio
# top 5 brands by total quantity sold - adidas,mi, lenovo, puma, ikea
# bottom 5 brands by total quantity sold - lg, milton, samsung, maybelline, sony

sales_by_brand.sort_values("quantity", ascending=False).head()
sales_by_brand['brand'] = sales_by_brand['brand'].str.strip().str.lower()
sales_by_brand.nunique()
sales_by_brand.sort_values("profit_x", ascending=False).head(10)
products['brand'] = products['brand'].str.strip().str.capitalize()
# channel 
channel_sales = sales_data.groupby('channel', as_index=False)[['sales', 'discount', 'quantity','profit_x','cost_price']].sum()
channel_sales
# Market place have the highest sales amount, with highest discount offering 
# store has least sales and discont offers 



In [ ]:
products.to_csv("C:/Users/abhis/OneDrive/Documents/abhishek/Data_Analyst_Abhishek/data_analyst/python_data/practice/enterprise ecommerce project/data/products.csv", index=False)

In [ ]:
# customer sales analysis
customers_sales = pd.merge(
    customers,
    orders,
    how='left',
    on='customer_id'
)
customers_sales.columns
customers_sales.nunique()
customers_sales.shape
customer_gender_sales = customers_sales.groupby('gender' , as_index=False)[['sales', 'discount','quantity','profit']].sum().sort_values("sales", ascending=False)
customer_gender_sales
# males have more sales, quantity and profit than female
# age based result
customer_age_based_sales = customers_sales.groupby("age_group", as_index=False)[['sales', 'discount','quantity','profit']].sum().sort_values("sales",ascending=False)
customer_age_based_sales.sort_values("profit", ascending=False)

# calculate profit contribution %
customer_age_based_sales['profit_contribution_pct'] = ((customer_age_based_sales['profit']/customer_age_based_sales['profit'].sum())*100).round(2)

customer_age_based_sales.sort_values("profit_contribution_pct", ascending=False)

# age group of 36-45 has the highest contributions in profit, with 21.4% 
# its allover the highest metrics in sales, discount, quantity and profit as well 
# segment
customer_segment_sales = customers_sales.groupby('customer_segment', as_index=False)[['sales', 'discount','quantity','profit','profit_contribution_pct']].sum().sort_values("sales",ascending=False)
customer_segment_sales

# home office has the highest profit margin with 24.8% and the lowest sales 
# state and city
state_and_city_sales = customers_sales.groupby(['state','city'], as_index=False)[['sales', 'discount','quantity','profit','profit_contribution_pct']].sum().sort_values("profit_contribution_pct",ascending=True)
state_and_city_sales.head()

# top city is kolkata, with 10.5 % profit contribution, sales too
# best state is west bengal with highest profit contribution 10.51% and sales
# tamil nadu has the lowest profit with 8.3% contribution
# pune and nagpur has the lowest profit contribution with 2.37 , 2.93 % only



In [ ]:
# Monthly Sales


(178576, 25)

,customer_segment,sales,discount,quantity,profit,profit_contribution_pct
1,Corporate,3.124684e+09,1361910,135961,1.932283e+08,24.19
3,Small Business,3.053422e+09,1331695,132434,2.000364e+08,24.79
0,Consumer,2.989046e+09,1314460,130797,1.861140e+08,23.02
2,Home Office,2.921219e+09,1278680,126987,1.920891e+08,24.80


,city,sales,discount,quantity,profit,profit_contribution_pct
17,Pune,3.988464e+08,178095,17743,18452400.99,2.37
14,Nagpur,3.630063e+08,156150,15808,24928031.10,2.93
12,Mumbai,4.552680e+08,200055,19813,26015795.80,3.23
10,Lucknow,3.997572e+08,178625,17528,22619962.60,3.24
8,Kanpur,4.052465e+08,178910,17720,25780790.40,3.42


In [ ]:
# #######################################################
# How would you replace missing values with the average sales?
#  first i took the median value of it do it gives me the avg median value of sales then i use the fill na to replace the naa value with median value
sales = sales.fillna(sales['Sales'].median())
# What is the difference between fillna(), replace(), and interpolate()?
# fillna is used to only replace the nan value , replace() is used to replace any value with it, interpolate() is used to change in the string value
# Which method would you choose in a sales dataset and why?
# i used fillna because there is nan value are the missing one so its directly reach to them , and median for avg sales value because i see the data is not have a lenear pattern if they have a linear pattern i will use mean at that time

#### 
# Why did you choose median instead of mean?
# i choose median instead of mean because the pattern of data is not linear to me it can be have the outliers thats why i prefer median when i think there is outlier because its elemenate them by itself i dont need to collect them 

# In what situations would mean actually be a better choice?
# when the data has no outlier then mean is good choice then it gives the correct answer for thesis

# Can you write the correct code to fill only the Sales column with the mean?
sales['sales'] = sales.fillna(sales['sales'].mean())
# What does interpolate() actually do?
# sorry i dont used it 
# Give one real-world example where you would use it.
# sorry i dont used it still but i will researched about it future
# Imagine you have 20 million rows of sales data.

# Would you still use fillna()?
#  yes i used the fillna() but with different approach first i remove the dupplicate values, keep the groupby averages for missing value i know it takes some times but it gives most closed results for the averages 
# Would you use pandas or another framework?
# sorry i dont know about it
# How would you optimize the process?
# delete the dupplicate values, taking the group by averages to get the closet result for analysis 

# ### 
# Write the code to merge these two DataFrames.
customer_orders = pd.merge(Orders, Customers, how='inner',on='CustomerID')
# Find the total amount spent by each customer.
customer_spend = customer_orders.groupby(['CustomerID','Name'], as_index=False)['Amount'].sum().sort_values('Amount', ascending=False)
# Which is faster for aggregation on large datasets: groupby() or a Python for loop? Why?
# groupby is fatest because it runs on specific conditions with a specific dataset in it, for loop is also a good choice but we need to add more lines of code in for loop but groupby has minimum words for code easy to understand and keeps the logic correct
# What is the difference between apply(), map(), and applymap() (or DataFrame.map() in newer pandas versions)?
# apply() is used to apply the conditions on particular dataframes, but map() is used to check the key value is in the dataframe or not it helps to get the data like dict, when we need to match the 1 data with its other row data map() is very useful , when we need to apply custom lamda functions on dataframe apply() is usefu lfor that 
# applymap is somethnig i didnt used so i cant tell the everything but in my knowledge it not only map the data with  key value but it also used when we need to functions on key value pair 

